In [1]:
import torch
import gc
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from datasets import load_dataset
from datasets import Dataset

import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)
from transformers import AutoConfig


c:\Users\Usuario\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [3]:
dataset = load_dataset("imdb")

In [4]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_MODEL_LENGTH = 2048
MAX_LENGTH = 2048

In [5]:
def build_prompt(review):

    prompt = f"""Review: {review}

Sentiment:"""

    return prompt


def build_full_text(review, label):

    sentiment = (
        "positive"
        if label == 1
        else "negative"
    )

    full_text = f"""Review: {review}

Sentiment: {sentiment}"""

    return full_text


def preprocess_dataset(dataset, tokenizer):

    input_ids_list = []
    attention_masks_list = []
    labels_list = []
    true_labels_list = []
    texts_list = []

    for example in dataset:

        review = example["text"]
        label  = example["label"]

        full_text  = build_full_text(review, label)
        prompt_ids = tokenizer(
            build_prompt(review), add_special_tokens=False
        )["input_ids"]
        prompt_len = len(prompt_ids)

        full = tokenizer(full_text, add_special_tokens=False)
        input_ids      = full["input_ids"]
        attention_mask = [1] * len(input_ids)
        labels = [
            -100 if i < prompt_len else tok
            for i, tok in enumerate(input_ids)
        ]

        if len(input_ids) > MAX_LENGTH:
            input_ids      = input_ids[:MAX_LENGTH]
            attention_mask = attention_mask[:MAX_LENGTH]
            labels         = labels[:MAX_LENGTH]
        else:
            pad_len        = MAX_LENGTH - len(input_ids)
            input_ids      += [tokenizer.pad_token_id] * pad_len
            attention_mask += [0] * pad_len
            labels         += [-100] * pad_len

        true_label = "positive" if label == 1 else "negative"

        input_ids_list.append(input_ids)
        attention_masks_list.append(attention_mask)
        labels_list.append(labels)
        true_labels_list.append(true_label)
        texts_list.append(review)

    return Dataset.from_dict({
        "input_ids":      input_ids_list,
        "attention_mask": attention_masks_list,
        "labels":         labels_list,
        "true_label":     true_labels_list,
        "texts":          texts_list,
    })


In [6]:
# Model tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [7]:
def fits_model(example):

    text = build_full_text(
        example["text"],
        example["label"]
    )

    tokens = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    return len(tokens) <= MAX_MODEL_LENGTH

In [8]:
BATCH_SIZE  = 1
N_PER_CLASS = 100  # 100 neg + 100 pos = 200 total

filtered_test = dataset["test"].filter(fits_model)
test_dataset  = preprocess_dataset(filtered_test, tokenizer)

random.seed(42)
pos_idx  = [i for i in range(len(test_dataset)) if test_dataset[i]["true_label"] == "positive"]
neg_idx  = [i for i in range(len(test_dataset)) if test_dataset[i]["true_label"] == "negative"]
eval_idx = random.sample(pos_idx, N_PER_CLASS) + random.sample(neg_idx, N_PER_CLASS)

test_subset = test_dataset.select(eval_idx)
test_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "true_label", "texts"])
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False)

eval_texts  = [test_subset[i]["texts"]      for i in range(len(test_subset))]
eval_labels = [test_subset[i]["true_label"] for i in range(len(test_subset))]
print(f"Eval: {len(eval_idx)} examples | "
      f"{eval_labels.count('positive')} pos, {eval_labels.count('negative')} neg")


Eval: 200 examples | 100 pos, 100 neg


In [9]:
CHECKPOINT_DIR = "../checkpoints/imdb"

config_11 = AutoConfig.from_pretrained(MODEL_NAME, local_files_only=True)
config_11.num_hidden_layers = 11


def load_model(n_layers, ckpt_name):
    if n_layers == 22:
        m = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, local_files_only=True, attn_implementation="eager"
        )
    else:
        m = AutoModelForCausalLM.from_config(config_11, attn_implementation="eager")
    m.load_state_dict(torch.load(f"{CHECKPOINT_DIR}/{ckpt_name}", map_location="cpu"))
    m.eval()
    return m


MODELS = {
    "teacher":       load_model(22, "best_teacher_model.pt"),
    "pure_student":      load_model(11, "best_pure_student_model.pt"),
}
MODEL_LABELS = {
    "teacher":       "P (teacher)",
    "pure_student":      "S3"
}
print("Loaded:", list(MODELS.keys()))


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3192.94it/s]


Loaded: ['teacher', 'pure_student']


# Métricas de comparación

In [10]:
def frobenius_difference(
    x: torch.Tensor,
    y: torch.Tensor
):
    return torch.norm(
        x - y,
        p="fro"
    )


def cosine_similarity_tensor(
    x: torch.Tensor,
    y: torch.Tensor,
    eps: float = 1e-8
):
    x_flat = x.reshape(-1)
    y_flat = y.reshape(-1)

    similarity = F.cosine_similarity(
        x_flat.unsqueeze(0),
        y_flat.unsqueeze(0),
        dim=1,
        eps=eps
    )

    return similarity.squeeze()

def js_divergence_attention(
    attn_teacher: torch.Tensor,
    attn_student: torch.Tensor,
    eps: float = 1e-8
):
    # normalizar filas
    p = attn_teacher / (
        attn_teacher.sum(dim=-1, keepdim=True)
        + eps
    )

    q = attn_student / (
        attn_student.sum(dim=-1, keepdim=True)
        + eps
    )

    m = 0.5 * (p + q)

    kl_pm = torch.sum(
        p * torch.log(
            (p + eps) / (m + eps)
        ),
        dim=-1
    )

    kl_qm = torch.sum(
        q * torch.log(
            (q + eps) / (m + eps)
        ),
        dim=-1
    )

    jsd_rows = 0.5 * (
        kl_pm + kl_qm
    )

    return jsd_rows.mean()

def linear_cka(X, Y):
    """Linear CKA between activation matrices X:[n,d1] and Y:[n,d2]."""
    X = X.float();  Y = Y.float()
    n = X.shape[0]
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    K = X @ X.T;  L = Y @ Y.T
    H = torch.eye(n, device=X.device) - torch.ones(n, n, device=X.device) / n
    Kc = H @ K @ H;  Lc = H @ L @ H
    hsic_kl = (Kc * Lc).sum()
    hsic_kk = (Kc * Kc).sum()
    hsic_ll = (Lc * Lc).sum()
    return (hsic_kl / (torch.sqrt(hsic_kk * hsic_ll) + 1e-10)).item()


def linear_cka(X, Y):
    """Linear CKA between activation matrices X:[n,d1] and Y:[n,d2]."""
    X = X.float();  Y = Y.float()
    n = X.shape[0]
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    K = X @ X.T;  L = Y @ Y.T
    H = torch.eye(n, device=X.device) - torch.ones(n, n, device=X.device) / n
    Kc = H @ K @ H;  Lc = H @ L @ H
    hsic_kl = (Kc * Lc).sum()
    hsic_kk = (Kc * Kc).sum()
    hsic_ll = (Lc * Lc).sum()
    return (hsic_kl / (torch.sqrt(hsic_kk * hsic_ll) + 1e-10)).item()


# Rango efectivo

In [11]:
def effective_rank_participation_ratio(
    x: torch.Tensor,
    eps: float = 1e-12
):

    x = x.float()

    singular_values = (
        torch.linalg.svdvals(x)
    )

    power = singular_values**2

    numerator = (
        power.sum()**2
    )

    denominator = (
        (power**2).sum()
        + eps
    )

    rank_eff = (
        numerator
        / denominator
    )

    return rank_eff


def effective_rank_entropy(
    x: torch.Tensor,
    eps: float = 1e-12
):

    x = x.float()

    singular_values = (
        torch.linalg.svdvals(x)
    )

    p = singular_values / (
        singular_values.sum()
        + eps
    )

    entropy = -torch.sum(
        p * torch.log(p + eps)
    )

    rank_eff = torch.exp(entropy)

    return rank_eff

# Atención promedio

In [12]:
def compute_average_attentions(
    model,
    dataloader,
    device,
    max_batches=None,
    print_every=50
):

    model.eval()

    n_layers = (
        model.config.num_hidden_layers
    )

    attention_sums = [

        torch.zeros(
            (2048, 2048),
            dtype=torch.float32,
            device="cpu"
        )

        for _ in range(n_layers)
    ]

    n_examples = 0

    with torch.no_grad():

        for batch_idx, batch in enumerate(
            dataloader
        ):

            if (
                max_batches is not None
                and batch_idx >= max_batches
            ):
                break

            input_ids = batch[
                "input_ids"
            ].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_attentions=True
            )

            attentions = (
                outputs.attentions
            )

            batch_size = (
                input_ids.size(0)
            )

            for layer_idx in range(
                n_layers
            ):

                attn = attentions[
                    layer_idx
                ]

                # (B, H, S, S)
                # promedio heads
                attn = attn.mean(
                    dim=1
                )

                # (B, S, S)
                attn = (
                    attn
                    .float()
                    .cpu()
                )

                # suma batch
                attn_sum = attn.sum(
                    dim=0
                )

                attention_sums[
                    layer_idx
                ] += attn_sum

            n_examples += batch_size

            if (
                batch_idx
                % print_every
                == 0
            ):

                print(
                    f"Batch "
                    f"{batch_idx} | "
                    f"Examples "
                    f"{n_examples}"
                )

            del (
                input_ids,
                attention_mask,
                outputs,
                attentions,
                attn
            )

            gc.collect()

            torch.cuda.empty_cache()

    attention_means = [

        attn_sum / n_examples

        for attn_sum
        in attention_sums
    ]

    return attention_means

In [13]:
def compute_hidden_states(model, prompts, tokenizer, device, n=100, max_seq=512):
    """
    Returns [n_layers+1, n, hidden_dim] – last real-token hidden vector per layer.
    """
    model.eval()
    all_hidden = []
    with torch.no_grad():
        for prompt in prompts[:n]:
            enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                            max_length=max_seq, add_special_tokens=False)
            input_ids = enc["input_ids"].to(device)
            attn_mask = enc["attention_mask"].to(device)
            last_idx  = int(attn_mask[0].nonzero()[-1].item())
            out = model(input_ids=input_ids, attention_mask=attn_mask,
                        output_hidden_states=True)
            hidden = torch.stack(
                [h[0, last_idx, :] for h in out.hidden_states]
            ).cpu()  # [n_layers+1, hidden_dim]
            all_hidden.append(hidden)
            del out;  torch.cuda.empty_cache()
    return torch.stack(all_hidden, dim=1)  # [n_layers+1, n, hidden_dim]


def compute_head_avg_patterns(model, prompts, tokenizer, device, n=50, max_seq=128):
    """
    Returns [n_layers, n_heads, max_seq] – avg attention from last token per head.
    """
    model.eval()
    n_layers = model.config.num_hidden_layers
    n_heads  = model.config.num_attention_heads
    sums  = torch.zeros(n_layers, n_heads, max_seq, dtype=torch.float32)
    count = 0
    with torch.no_grad():
        for prompt in prompts[:n]:
            enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                            max_length=max_seq, add_special_tokens=False)
            input_ids = enc["input_ids"].to(device)
            attn_mask = enc["attention_mask"].to(device)
            seq_len   = input_ids.size(1)
            last_idx  = int(attn_mask[0].nonzero()[-1].item())
            out = model(input_ids=input_ids, attention_mask=attn_mask,
                        output_attentions=True)
            for l in range(n_layers):
                # [1, n_heads, S, S] -> [n_heads, seq_len]
                ha = out.attentions[l][0, :, last_idx, :seq_len].float().cpu()
                padded = torch.zeros(n_heads, max_seq)
                padded[:, :seq_len] = ha
                sums[l] += padded
            count += 1
            del out;  torch.cuda.empty_cache()
    return sums / count  # [n_layers, n_heads, max_seq]


## Teacher

In [14]:
# Central attention computation for all 5 models
model_attentions = {}
for name, model in MODELS.items():
    print(f"Computing attentions for {MODEL_LABELS[name]}...")
    model.to(device)
    model_attentions[name] = compute_average_attentions(model, test_loader, device)
    model.cpu()
    torch.cuda.empty_cache()
    gc.collect()

# 22-layer teacher attentions (needed by teacher_avg_attentions + teacher_rollouts)
teacher_attentions = model_attentions["teacher"]

# Aliases — existing pairwise comparison cells use these names unchanged
pure_student_attentions      = model_attentions["pure_student"]

print("All attention maps computed.")


Computing attentions for P (teacher)...
Batch 0 | Examples 1
Batch 50 | Examples 51
Batch 100 | Examples 101
Batch 150 | Examples 151
Computing attentions for S3...
Batch 0 | Examples 1
Batch 50 | Examples 51
Batch 100 | Examples 101
Batch 150 | Examples 151
All attention maps computed.


In [15]:
teacher_avg_attentions = [
    (teacher_attentions[2*i] + teacher_attentions[2*i + 1]) / 2
    for i in range(11)
]

In [16]:
def rollout_pair(A1, A2):
    I = torch.eye(A1.shape[-1], device=A1.device)
    
    # agregar residual y renormalizar
    A1 = A1 + I
    A1 = A1 / A1.sum(dim=-1, keepdim=True)
    
    A2 = A2 + I
    A2 = A2 / A2.sum(dim=-1, keepdim=True)
    
    return A1 @ A2

teacher_rollouts = [rollout_pair(teacher_attentions[2*i], teacher_attentions[2*i+1]) for i in range(11)]

# Teacher - Pure student comparison

In [17]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (battn, tattn) in enumerate(zip(pure_student_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(battn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (battn, tattn) in enumerate(zip(pure_student_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(battn, tattn))

Similitud coseno con promedio de atenciones
tensor(0.9397)
tensor(0.6622)
tensor(0.6426)
tensor(0.6101)
tensor(0.4715)
tensor(0.3850)
tensor(0.2192)
tensor(0.2206)
tensor(0.2536)
tensor(0.2476)
tensor(0.1838)


Similitud coseno con Attention Rollout
tensor(0.2853)
tensor(0.5789)
tensor(0.6118)
tensor(0.5997)
tensor(0.4934)
tensor(0.4170)
tensor(0.2745)
tensor(0.3135)
tensor(0.3324)
tensor(0.3196)
tensor(0.2729)


In [18]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (battn, tattn) in enumerate(zip(pure_student_attentions, teacher_avg_attentions)):
    print(frobenius_difference(battn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (battn, tattn) in enumerate(zip(pure_student_attentions, teacher_rollouts)):
    print(frobenius_difference(battn, tattn))

Distancia Frobenius con promedio de atenciones
tensor(1.2015)
tensor(16.8296)
tensor(32.7808)
tensor(37.2720)
tensor(24.9249)
tensor(21.6115)
tensor(17.9569)
tensor(17.6494)
tensor(19.0600)
tensor(19.9401)
tensor(21.1148)


Distancia Frobenius con Attention Rollout
tensor(11.3397)
tensor(19.8765)
tensor(27.6835)
tensor(30.5788)
tensor(23.8355)
tensor(21.9973)
tensor(19.6687)
tensor(19.4344)
tensor(20.3861)
tensor(20.9275)
tensor(21.7369)


In [19]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (battn, tattn) in enumerate(zip(pure_student_attentions, teacher_avg_attentions)):
    print(js_divergence_attention(battn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (battn, tattn) in enumerate(zip(pure_student_attentions, teacher_rollouts)):
    print(js_divergence_attention(battn, tattn))

JS promediada por fila con promedio de atenciones
tensor(0.0156)
tensor(0.1253)
tensor(0.3234)
tensor(0.4132)
tensor(0.2987)
tensor(0.2768)
tensor(0.2728)
tensor(0.2749)
tensor(0.3123)
tensor(0.3274)
tensor(0.3206)


JS promediada por fila con Attention Rollout
tensor(0.1107)
tensor(0.2485)
tensor(0.4030)
tensor(0.4706)
tensor(0.3624)
tensor(0.3377)
tensor(0.3212)
tensor(0.3192)
tensor(0.3522)
tensor(0.3642)
tensor(0.3666)


In [20]:
# Participation ratio
print("Participation ratio")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (battn, tattn, tattn2) in enumerate(zip(pure_student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_participation_ratio(battn):4f} | {effective_rank_participation_ratio(tattn):4f} | {effective_rank_participation_ratio(tattn2):4f}")

Participation ratio
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 2.658184 | 3.261388 | 399.641205
1 | 1.791254 | 1.010796 | 1.863250
2 | 1.501657 | 1.000376 | 1.314646
3 | 1.814849 | 1.000595 | 1.287964
4 | 3.643436 | 1.003630 | 1.549976
5 | 5.444178 | 1.005531 | 1.718067
6 | 16.249025 | 1.014576 | 2.142127
7 | 25.819513 | 1.016638 | 2.155190
8 | 12.163080 | 1.012526 | 1.972973
9 | 11.224791 | 1.008846 | 1.895686
10 | 19.548100 | 1.015690 | 1.878476


In [21]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (battn, tattn, tattn2) in enumerate(zip(pure_student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_entropy(battn):4f} | {effective_rank_entropy(tattn):4f} | {effective_rank_entropy(tattn2):4f}")

Entropy based effective rank
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 109.979767 | 309.320038 | 2026.606689
1 | 186.378647 | 42.224606 | 1819.434326
2 | 57.598793 | 5.203577 | 1667.716187
3 | 276.581665 | 6.992298 | 1650.066284
4 | 415.579742 | 20.366531 | 1761.163696
5 | 456.668243 | 26.349777 | 1797.661987
6 | 512.440369 | 66.281708 | 1849.150146
7 | 568.967041 | 84.208687 | 1850.003052
8 | 530.855774 | 64.056908 | 1832.478882
9 | 505.231171 | 45.523666 | 1823.890381
10 | 544.828125 | 129.148560 | 1819.147217
